# 04 ? Chronos-2 + equation head com curriculum

Este notebook especifica o experimento com o gerador tsfmxai. O Chronos-2 fornece features por s?rie; heads auxiliares predizem o embedding congelado do encoder canonicalizador de skeleton e o desvio padr?o da inova??o. O curr?culo aumenta simultaneamente canais e opera??es em limites de batches fixos.

## Registro

- Estado: defini??o execut?vel de agenda e losses auxiliares; modelo Chronos-2 precisa de wrapper de treino customizado antes da execu??o completa.
- Inicializa??o: mesmo checkpoint Chronos-2 dos experimentos de forecasting.
- Teacher: repo/revision/variant do Hugging Face precisam ser fixados no notebook 03. Professor prev? skeleton sem coeficientes e permanece congelado.
- Exposi??o alvo: mesmas atualiza??es e batches do experimento sem equation head; custo extra de heads/teacher reportado.
- Stage muda por n?mero fixo de updates, n?o por quantidade vari?vel de dados produzidos.


Checkpoint Chronos-2 fixado em `29ec3766d36d6f73f0696f85560a422f50e8498c`. O teacher skeleton publicado em `2fcdcc88b41125bcfd92953c0bb8a668a603d167` tem somente uma s?rie no vocabul?rio (37 tokens); o teacher single-series pode ser usado somente com topologia independente (cada equa??o depende apenas de si pr?pria, relabel local). Depend?ncias cruzadas exigem teacher de seis s?ries. Para comparar curriculum/head, controle e tratamento devem usar exatamente a mesma topologia independente.

## Reproduction instructions

Use Python 3.11 from the repository root. Install the project dependencies with `python -m pip install -e ".[learning,dev,release]"`, then execute this notebook with:

`python -m nbconvert --to notebook --execute --inplace experiments/generator_model_evaluation/04_chronos2_equation_head_curriculum.ipynb`

The Chronos-2 checkpoint used by this experiment is pinned to `amazon/chronos-2@29ec3766d36d6f73f0696f85560a422f50e8498c`; the skeleton teacher, where used, is pinned to `mateuswiteck/tsfmxai-canonicalization-encoder-decoder` variant `transformer_skeleton` at `2fcdcc88b41125bcfd92953c0bb8a668a603d167`. The notebook records its own seeds, settings, observed result, and current limitations.


In [1]:
from dataclasses import dataclass
import torch
from torch import nn
import torch.nn.functional as F

@dataclass(frozen=True)
class CurriculumStage:
    start_update: int
    channels: int
    max_operators: int
    max_degree: int

STAGES = (
    CurriculumStage(0, 2, 1, 1),
    CurriculumStage(2_000, 4, 3, 2),
    CurriculumStage(4_000, 6, 5, 3),
)
TOTAL_UPDATES = 6_000
DEPENDENCY_MODE = 'independent'  # required by the pinned single-series skeleton teacher
LEARNING_RATE = 1e-6
BATCH_EXAMPLES = 8
CONTEXT_LENGTH = 512
PREDICTION_LENGTH = 64
assert DEPENDENCY_MODE == 'independent'
assert STAGES[0].start_update == 0
assert all(a.start_update < b.start_update for a,b in zip(STAGES,STAGES[1:]))

def curriculum_for_update(update):
    if not 0 <= update < TOTAL_UPDATES:
        raise IndexError(update)
    selected=STAGES[0]
    for stage in STAGES[1:]:
        if stage.start_update > update: break
        selected=stage
    return selected
print([curriculum_for_update(i) for i in (0,1999,2000,3999,4000,5999)])

[CurriculumStage(start_update=0, channels=2, max_operators=1, max_degree=1), CurriculumStage(start_update=0, channels=2, max_operators=1, max_degree=1), CurriculumStage(start_update=2000, channels=4, max_operators=3, max_degree=2), CurriculumStage(start_update=2000, channels=4, max_operators=3, max_degree=2), CurriculumStage(start_update=4000, channels=6, max_operators=5, max_degree=3), CurriculumStage(start_update=4000, channels=6, max_operators=5, max_degree=3)]


## Heads e fun??o objetivo

O backbone deve expor uma representa??o `[linhas_canal, dimens?o]` associada ao contexto de cada canal. A interface concreta depende da revis?o do Chronos-2; n?o se assume um nome interno de m?dulo. `skeleton_head` produz a dimens?o do embedding teacher. `noise_head` prev? `log(std)` para manter positividade ap?s exponencia??o.

`teacher_embedding` ? produzido uma vez por mecanismo com `CanonicalEncoder` congelado. Normalize o target por norma L2 e minimize MSE/cosseno; guardar valida??o para escolher a loss. Noise usa erro em log-escala porque os n?veis podem variar multiplicativamente. Forecast loss permanece exatamente a loss oficial do Chronos-2.


In [2]:
class AuxiliaryHeads(nn.Module):
    def __init__(self, backbone_dim, teacher_dim):
        super().__init__()
        self.skeleton = nn.Sequential(nn.LayerNorm(backbone_dim), nn.Linear(backbone_dim, teacher_dim))
        self.log_noise_std = nn.Sequential(nn.LayerNorm(backbone_dim), nn.Linear(backbone_dim, 1))

    def forward(self, channel_features):
        return self.skeleton(channel_features), self.log_noise_std(channel_features).squeeze(-1)

def auxiliary_losses(predicted_embedding, teacher_embedding, predicted_log_std, target_std):
    teacher = F.normalize(teacher_embedding.detach(), dim=-1)
    predicted = F.normalize(predicted_embedding, dim=-1)
    embedding_loss = 1.0 - F.cosine_similarity(predicted, teacher, dim=-1).mean()
    eps = torch.finfo(target_std.dtype).eps
    noise_loss = F.smooth_l1_loss(predicted_log_std, torch.log(target_std.clamp_min(eps)))
    return {"embedding": embedding_loss, "innovation_log_std": noise_loss}

# Full loss in the custom trainer:
# total = chronos_forecast_loss + lambda_embedding * losses['embedding']
#       + lambda_noise * losses['innovation_log_std']

## Dataset e execu??o

Em cada update, determine o est?gio pela fun??o acima, gere oito exemplos com exatamente o n?mero de canais daquele est?gio, adapte cada canal para linhas do Chronos-2 e mantenha `group_ids`. Ap?s 6.000 updates, exemplos de cada est?gio ter?o recebido 2.000 updates. Use o mesmo horizonte/contexto e o mesmo or?amento da condi??o sem heads.

O professor deve ser executado em lotes antes ou durante a prepara??o de dados, nunca um processo CUDA duplicado em cada DataLoader worker. Cachear embeddings por revis?o do teacher e hash da entrada AST; como o encoder atual consome valores num?ricos, incluir esses valores no hash at? que independ?ncia do skeleton esteja validada.

Valida??o fixa inclui exemplos dos tr?s n?veis de dificuldade e mecanismos n?o vistos. Reportar m?tricas por est?gio e pelo subconjunto de dificuldade m?xima para detectar esquecimento. Separar sistemas/skeletons inteiros entre treino, valida??o e teste.

## Condi??es que bloqueiam um treino cient?fico

1. Resolvido para topologia independente: teacher single-series relabela cada equa??o self-only; cross-series continua exigindo teacher de seis s?ries.
2. Implementar wrapper do Chronos-2 que retorne features por canal e otimize as losses conjunta e oficial.
3. Resolver limites do vocabul?rio: o teacher precisa suportar seis s?ries e a gram?tica/lags do est?gio final.
4. Validar estabilidade do gerador em cada est?gio e definir pol?tica de trajet?rias inv?lidas.
5. Congelar pesos das losses na valida??o piloto e usar mesmo n?mero de updates nos controles.
6. A GPU precisa estar dispon?vel para o fine-tuning planejado; este ambiente atual ? CPU-only.

O curr?culo aumenta canais e operadores nos mesmos marcos. Para isolar o efeito da ordem, comparar depois com controle sem curr?culo que veja a mesma mistura total de est?gios embaralhada desde o in?cio.

Chronos-2 fixado: `amazon/chronos-2` @ `29ec3766d36d6f73f0696f85560a422f50e8498c`. Usar `dependency_probs={'independent':1.0}` nos dois bra?os. N?o incluir liga??es cruzadas at? existir teacher compat?vel.